In [75]:
import numpy as np
from PIL import Image
import wave
from scipy.io.wavfile import write as write_wav
import librosa
import pywt

from glob import glob
import  re

In [76]:
pattern = re.compile(r'.*\.(png|jpg|jpeg)$', re.IGNORECASE)
images = [file for file in glob('*') if pattern.match(file)]
sounds = glob("*.wav")

images, sounds


(['mixkit-classic-alarm-995.png', 'test3.jpg', 'image_with_sound_dwt.png'],
 ['mixkit-classic-alarm-995.wav', 'extracted_sound_dwt.wav'])

In [77]:
image = Image.open(images[1])
image_data = np.array(image)
sound_data, sample_rate = librosa.load(sounds[0])

output_image_path = "image_with_sound_dwt.png"
output_sound_path = "extracted_sound_dwt.wav"
sample_rate

22050

In [78]:
coeffs = pywt.wavedec(sound_data, wavelet='haar', level=2)
flattened_coeffs = np.hstack([c.flatten() for c in coeffs])


flat_image_data = image_data.flatten()
if len(flattened_coeffs) > len(flat_image_data):
    raise ValueError("DWT coefficients are too large to embed in the given image.")

In [79]:
flat_image_data[:len(flattened_coeffs)] = (
    flat_image_data[:len(flattened_coeffs)] & 0xFE | (np.round(flattened_coeffs).astype(int) & 1)
)
embedded_image_data = flat_image_data.reshape(image_data.shape)

embedded_image = Image.fromarray(embedded_image_data)
embedded_image.save(output_image_path)
print(f"Sound embedded successfully into {output_image_path} using DWT.")

Sound embedded successfully into image_with_sound_dwt.png using DWT.


In [80]:
image_with_sound = Image.open(output_image_path)
image_with_sound_data = np.array(image_with_sound)

flat_image_with_sound_data = image_with_sound_data.flatten()
extracted_coeff_bits = flat_image_with_sound_data[:len(flattened_coeffs)] & 1

In [81]:
extracted_coeffs = np.array(extracted_coeff_bits, dtype=float)
coeff_shapes = [c.shape for c in coeffs]
reconstructed_coeffs = []
start_idx = 0

for shape in coeff_shapes:
    size = np.prod(shape)
    reconstructed_coeffs.append(extracted_coeffs[start_idx:start_idx + size].reshape(shape))
    start_idx += size

reconstructed_sound = pywt.waverec(reconstructed_coeffs, wavelet='haar')

In [82]:
write_wav(output_sound_path, sample_rate, (reconstructed_sound * 32767).astype(np.int16))
print(f"Sound extracted and saved to {output_sound_path} using DWT.")

Sound extracted and saved to extracted_sound_dwt.wav using DWT.
